# Day 3b — Spam Filter, Step by Step
**30-Day ML & AI Mastery Plan · Shubham**

The same Naive Bayes spam filter as Day 3, but slowed down. We use only **4 training emails**, and every cell prints its intermediate values so you can check each number by hand.

The whole idea in one sentence: **an email is spam if its words appear more often in spam than in normal mail.** Everything below is just making that sentence precise.

In [1]:
import re, math
from collections import Counter

## Step 1 — The training data
Four emails a person has already labelled. "ham" means a normal, wanted email.

In [2]:
train = [
    ("Win a FREE prize!",        "spam"),
    ("Free cash, click now",     "spam"),
    ("Team meeting on Friday",   "ham"),
    ("Free lunch for the team",  "ham"),
]
for text, label in train:
    print(f"{label:>4} | {text}")

spam | Win a FREE prize!
spam | Free cash, click now
 ham | Team meeting on Friday
 ham | Free lunch for the team


## Step 2 — Turn text into words (tokenise)
A computer can't compare sentences, but it can count words. Three things happen:
1. `text.lower()` makes "FREE" and "Free" the same word.
2. `re.findall(r"[a-z]+", ...)` keeps runs of letters only, so "prize!" becomes "prize" and the comma after "cash" disappears.
3. Very common words ("a", "the", "for", "now") are removed. They appear everywhere, so they say nothing about spam.

In [3]:
STOPWORDS = {"a", "the", "to", "your", "you", "is", "for", "and", "of", "on", "by", "my", "me", "i", "we", "our", "from", "can", "be", "will", "has", "are", "here", "now"}

def tokenize(text):
    words = re.findall(r"[a-z]+", text.lower())
    return [w for w in words if w not in STOPWORDS]

# Watch each stage on one email
text = "Win a FREE prize!"
print("1. original:     ", text)
print("2. lower():      ", text.lower())
print("3. findall:      ", re.findall(r"[a-z]+", text.lower()))
print("4. no stopwords: ", tokenize(text))

1. original:      Win a FREE prize!
2. lower():       win a free prize!
3. findall:       ['win', 'a', 'free', 'prize']
4. no stopwords:  ['win', 'free', 'prize']


In [4]:
for text, label in train:
    print(f"{label:>4} | {text:<26} -> {tokenize(text)}")

spam | Win a FREE prize!          -> ['win', 'free', 'prize']
spam | Free cash, click now       -> ['free', 'cash', 'click']
 ham | Team meeting on Friday     -> ['team', 'meeting', 'friday']
 ham | Free lunch for the team    -> ['free', 'lunch', 'team']


## Step 3 — Count words in each class
`Counter` is a dictionary that counts. `.update(list)` adds 1 for every item in the list.

In [5]:
c = Counter()
c.update(["free", "cash"])
print("after first update: ", c)
c.update(["free"])
print("after second update:", c)

after first update:  Counter({'free': 1, 'cash': 1})
after second update: Counter({'free': 2, 'cash': 1})


In [6]:
counts = {"spam": Counter(), "ham": Counter()}
for text, label in train:
    counts[label].update(tokenize(text))

print("spam:", dict(counts["spam"]))
print("ham: ", dict(counts["ham"]))
print("a word never seen in a class counts as 0, not an error:", counts["ham"]["cash"])

spam: {'win': 1, 'free': 2, 'prize': 1, 'cash': 1, 'click': 1}
ham:  {'team': 2, 'meeting': 1, 'friday': 1, 'free': 1, 'lunch': 1}
a word never seen in a class counts as 0, not an error: 0


In [7]:
vocab = set(counts["spam"]) | set(counts["ham"])    # every distinct word seen anywhere
total = {label: sum(counts[label].values()) for label in counts}
print("vocabulary:", sorted(vocab), "->", len(vocab), "words")
print("total words per class:", total)

vocabulary: ['cash', 'click', 'free', 'friday', 'lunch', 'meeting', 'prize', 'team', 'win'] -> 9 words
total words per class: {'spam': 6, 'ham': 6}


## Step 4 — The prior: how common is spam overall?
Before reading a single word, what's the chance any email is spam? 2 of our 4 training emails are spam, so **P(spam) = 0.5**. In real life this might be 0.2 or 0.8. This is the **prior** from Bayes' theorem.

In [8]:
labels = [label for _, label in train]
prior = {label: labels.count(label) / len(labels) for label in counts}
print(prior)

{'spam': 0.5, 'ham': 0.5}


## Step 5 — Word probabilities, and why we add 1
**P(word | spam)** = "if I pick a random word from spam emails, how likely is it to be this word?"

Plain version: count / total. P("free" | spam) = 2 / 6 = 0.333.

**The problem:** "cash" never appeared in ham, so P("cash" | ham) = 0 / 6 = 0. Multiplying anything by 0 gives 0, so **one word could single-handedly decide the class**. Run the next cell to see it break.

In [9]:
def p_word_plain(word, label):
    return counts[label][word] / total[label]

email = ["free", "cash", "meeting"]   # 2 spammy words + 1 work word
for label in counts:
    score = prior[label]
    for w in email:
        score *= p_word_plain(w, label)
    print(f"{label:>4}: {score}")
print("-> both are 0. The model can't decide anything!")

spam: 0.0
 ham: 0.0
-> both are 0. The model can't decide anything!


**The fix, Laplace smoothing:** pretend every vocabulary word was seen **once more** than it really was, in every class.

**P(word | class) = (count + 1) / (total words in class + vocabulary size)**

We add the vocabulary size (9) to the bottom because we added 1 for each of the 9 words. That keeps all the probabilities in a class summing to 1.

In [10]:
V = len(vocab)
def p_word(word, label):
    return (counts[label][word] + 1) / (total[label] + V)

print(f"{'word':<8}{'spam count':>11}{'P(w|spam)':>14}{'ham count':>11}{'P(w|ham)':>14}")
for w in sorted(vocab):
    print(f"{w:<8}{counts['spam'][w]:>11}{counts['spam'][w]+1:>7}/{total['spam']+V} ={p_word(w,'spam'):.3f}"
          f"{counts['ham'][w]:>11}{counts['ham'][w]+1:>7}/{total['ham']+V} ={p_word(w,'ham'):.3f}")

print("\nprobabilities in each class sum to:", round(sum(p_word(w, 'spam') for w in vocab), 6), round(sum(p_word(w, 'ham') for w in vocab), 6))

word     spam count     P(w|spam)  ham count      P(w|ham)
cash              1      2/15 =0.133          0      1/15 =0.067
click             1      2/15 =0.133          0      1/15 =0.067
free              2      3/15 =0.200          1      2/15 =0.133
friday            0      1/15 =0.067          1      2/15 =0.133
lunch             0      1/15 =0.067          1      2/15 =0.133
meeting           0      1/15 =0.067          1      2/15 =0.133
prize             1      2/15 =0.133          0      1/15 =0.067
team              0      1/15 =0.067          2      3/15 =0.200
win               1      2/15 =0.133          0      1/15 =0.067

probabilities in each class sum to: 1.0 1.0


In [11]:
# The broken example again, now smoothed
for label in counts:
    score = prior[label]
    for w in email:
        score *= p_word(w, label)
    print(f"{label:>4}: {score:.6f}")
print("-> no more zeros, and spam wins (free + cash outweigh meeting)")

spam: 0.000889
 ham: 0.000593
-> no more zeros, and spam wins (free + cash outweigh meeting)


## Step 6 — Score a new email
Test email: **"Free cash prize"** → tokens `free, cash, prize`.

score(class) = P(class) × P(free | class) × P(cash | class) × P(prize | class)

This multiplication is where the **"naive"** assumption hides: we treat words as independent, as if seeing "free" says nothing about whether "prize" will appear too.

In [12]:
test = "Free cash prize"
tokens = tokenize(test)
print("tokens:", tokens, "\n")

raw = {}
for label in counts:
    score = prior[label]
    steps = [f"{prior[label]}"]
    for w in tokens:
        score *= p_word(w, label)
        steps.append(f"{counts[label][w]+1}/{total[label]+V}")
    raw[label] = score
    print(f"{label:>4}: {' x '.join(steps)} = {score:.7f}")

tokens: ['free', 'cash', 'prize'] 

spam: 0.5 x 3/15 x 2/15 x 2/15 = 0.0017778
 ham: 0.5 x 2/15 x 1/15 x 1/15 = 0.0002963


## Step 7 — Turn scores into a probability
The two scores aren't probabilities yet: they don't add up to 1. Divide each by their sum.

In [13]:
s = raw["spam"] + raw["ham"]
for label in raw:
    print(f"P({label} | email) = {raw[label]:.7f} / {s:.7f} = {raw[label]/s:.3f}")
print(f"\nThe spam score is {raw['spam']/raw['ham']:.0f} times the ham score -> SPAM")

P(spam | email) = 0.0017778 / 0.0020741 = 0.857
P(ham | email) = 0.0002963 / 0.0020741 = 0.143

The spam score is 6 times the ham score -> SPAM


## Step 8 — Why real code adds logs instead of multiplying
Real emails have hundreds of words. Multiplying hundreds of small numbers becomes too small for a computer to store and turns into exactly 0.0 (**underflow**).

Logs fix this because **log(a × b) = log(a) + log(b)**. Products become sums, and sums of moderate negative numbers never underflow. The ranking of the classes stays the same.

In [14]:
print("a long email, multiplied:", 0.01 ** 200)
print("the same email in logs:  ", 200 * math.log(0.01))

a long email, multiplied: 0.0
the same email in logs:   -921.0340371976182


In [15]:
log_score = {}
for label in counts:
    parts = [math.log(prior[label])] + [math.log(p_word(w, label)) for w in tokens]
    log_score[label] = sum(parts)
    print(f"{label:>4}: " + " + ".join(f"({p:.4f})" for p in parts) + f" = {log_score[label]:.4f}")

print("\nexp() undoes log:", {k: round(math.exp(v), 7) for k, v in log_score.items()}, "<- same as the raw scores in Step 6")

spam: (-0.6931) + (-1.6094) + (-2.0149) + (-2.0149) = -6.3324
 ham: (-0.6931) + (-2.0149) + (-2.7081) + (-2.7081) = -8.1242

exp() undoes log: {'spam': 0.0017778, 'ham': 0.0002963} <- same as the raw scores in Step 6


To convert log scores back to probabilities safely, subtract the largest score before `exp()`. The ratio between the classes doesn't change, but the numbers stay in a safe range. This is called the **log-sum-exp trick**.

In [16]:
m = max(log_score.values())
shifted = {k: v - m for k, v in log_score.items()}
ex = {k: math.exp(v) for k, v in shifted.items()}
z = sum(ex.values())
for k in log_score:
    print(f"{k:>4}: log {log_score[k]:.4f} -> shifted {shifted[k]:.4f} -> exp {ex[k]:.4f} -> P = {ex[k]/z:.3f}")

spam: log -6.3324 -> shifted 0.0000 -> exp 1.0000 -> P = 0.857
 ham: log -8.1242 -> shifted -1.7918 -> exp 0.1667 -> P = 0.143


## Step 9 — Explain the decision: which word pushed which way?
For each word, compare how likely it is in spam vs ham. **log(P(w|spam) / P(w|ham))**:
- positive → the word pushes toward spam
- negative → toward ham
- zero → no information

In [17]:
for w in tokens:
    ratio = p_word(w, "spam") / p_word(w, "ham")
    print(f"{w:>6}: P(w|spam)/P(w|ham) = {ratio:.2f}  log = {math.log(ratio):+.3f}  {'-> spam' if ratio > 1 else '-> ham'}")

  free: P(w|spam)/P(w|ham) = 1.50  log = +0.405  -> spam
  cash: P(w|spam)/P(w|ham) = 2.00  log = +0.693  -> spam
 prize: P(w|spam)/P(w|ham) = 2.00  log = +0.693  -> spam


## Step 10 — Try more emails, including a perfect tie

In [18]:
def predict_proba(text):
    toks = tokenize(text)
    ls = {label: math.log(prior[label]) + sum(math.log(p_word(w, label)) for w in toks) for label in counts}
    m = max(ls.values())
    ex = {k: math.exp(v - m) for k, v in ls.items()}
    z = sum(ex.values())
    return {k: v / z for k, v in ex.items()}

for t in ["Free cash prize", "Click to win cash", "Team lunch on Friday", "Free prize for the team", "Hello mom"]:
    p = predict_proba(t)
    verdict = "SPAM" if p["spam"] > 0.5 else "ham" if p["spam"] < 0.5 else "TIE"
    print(f"{verdict:>4}  P(spam)={p['spam']:.3f}  tokens={tokenize(t)}  | {t}")

SPAM  P(spam)=0.857  tokens=['free', 'cash', 'prize']  | Free cash prize
SPAM  P(spam)=0.889  tokens=['click', 'win', 'cash']  | Click to win cash
 ham  P(spam)=0.077  tokens=['team', 'lunch', 'friday']  | Team lunch on Friday
 TIE  P(spam)=0.500  tokens=['free', 'prize', 'team']  | Free prize for the team
 TIE  P(spam)=0.500  tokens=['hello', 'mom']  | Hello mom


- **"Free prize for the team"** is an exact tie: "free" and "prize" push toward spam by exactly as much as "team" pushes toward ham. More training data would break the tie.
- **"Hello mom"**: none of its words were in training, so the model falls back to the prior (0.5). With no evidence, it can only say what it believed before.

## You now understand the full algorithm
| Step | Code | Probability idea |
|---|---|---|
| Tokenise | `re.findall`, stopwords | turning text into countable events |
| Count | `Counter.update` | frequencies |
| Prior | `labels.count(c) / n` | P(class) |
| Likelihood | `(count + 1) / (total + V)` | P(word \| class) with Laplace smoothing |
| Score | prior × likelihoods | Bayes' theorem + the naive independence assumption |
| Logs | `math.log`, sum | avoids underflow |
| Normalise | divide by the sum | posterior probability that sums to 1 |

**Next:** go back to the Day 3 notebook, Part 8. The `NaiveBayesSpam` class is exactly these steps wrapped in methods, trained on 16 emails instead of 4.